# Corrected QUBO / Hamiltonian generator

This notebook is the Jupyter wrapper for `qubo_hamiltonian.py`.

Important: the mathematical logic lives in `qubo_hamiltonian.py`. This notebook deliberately calls that file instead of duplicating the QUBO code, so the notebook cannot accidentally regenerate the old all-to-all Hamiltonian.

Default target compression remains **30%** (`--target-compression 0.30`).


## What this notebook does

1. Checks that `qubo_hamiltonian.py` exists.
2. Runs the corrected sparse topology-aware QUBO/Hamiltonian generator.
3. Reads the generated metadata and energy check files.
4. Shows the selected pruning solution.

Generated files are written to `qubo_outputs/`.


In [1]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_DIR = Path.cwd()
SCRIPT_PATH = PROJECT_DIR / 'qubo_hamiltonian.py'
INPUT_PATH = PROJECT_DIR / 'cost_loss_table.csv'
OUTPUT_DIR = PROJECT_DIR / 'qubo_outputs'

print('Project directory:', PROJECT_DIR)
print('Script path:', SCRIPT_PATH)
print('Input CSV:', INPUT_PATH)
print('Output directory:', OUTPUT_DIR)

if not SCRIPT_PATH.exists():
    raise FileNotFoundError('Missing qubo_hamiltonian.py. Paste the corrected Python file into the project root first.')

if not INPUT_PATH.exists():
    fallback = OUTPUT_DIR / 'cost_loss_table.csv'
    if fallback.exists():
        INPUT_PATH = fallback
        print('Using fallback input CSV:', INPUT_PATH)
    else:
        raise FileNotFoundError('Missing cost_loss_table.csv in project root and qubo_outputs/.')


Project directory: c:\_projects\quantum_pruning
Script path: c:\_projects\quantum_pruning\qubo_hamiltonian.py
Input CSV: c:\_projects\quantum_pruning\cost_loss_table.csv
Output directory: c:\_projects\quantum_pruning\qubo_outputs


## Run corrected QUBO/Hamiltonian generation

This runs the corrected model:

\[
H(x)=\sum_i(\alpha L_i-\beta C_i)x_i+\rho\sum_{(i,j)\in E}w_{ij}x_ix_j
\]

The old global squared target formulation is not used as the default because it creates all-to-all ZZ couplings.


In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT_PATH),
    '--input', str(INPUT_PATH),
    '--outdir', str(OUTPUT_DIR),
    '--max-candidates', '10',
    '--formulation', 'sparse_topology',
    '--interaction-method', 'all_pairs',
    '--loss-metric', 'accuracy',
    '--rho-interaction', '0.90',
    '--measured-pairwise-csv', str(OUTPUT_DIR / 'pairwise_sensitivity.csv'),
    '--target-compression', '0.40',
]

print('Running command:')
print(' '.join(cmd))

result = subprocess.run(cmd, cwd=PROJECT_DIR, text=True, capture_output=True)
print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('QUBO/Hamiltonian generation failed.')


## Inspect generated metadata


In [3]:
metadata_path = OUTPUT_DIR / 'qubo_metadata.json'
complexity_path = OUTPUT_DIR / 'circuit_complexity_report.json'
energy_check_path = OUTPUT_DIR / 'qubo_energy_check.csv'

with metadata_path.open('r', encoding='utf-8') as f:
    metadata = json.load(f)

with complexity_path.open('r', encoding='utf-8') as f:
    complexity = json.load(f)

print('Formulation:', metadata.get('formulation'))
print('Target compression:', metadata.get('target_compression'))
print('Beta used:', metadata.get('beta'))
print('Beta auto calibrated:', metadata.get('beta_auto_calibrated'))
print('Interaction method:', metadata.get('interaction_method'))
print('Sparse ZZ terms:', complexity.get('zz_terms'))
print('Legacy all-to-all ZZ terms for same n:', complexity.get('legacy_all_to_all_zz_terms_for_same_n'))
print('ZZ reduction vs legacy all-to-all:', complexity.get('zz_reduction_vs_legacy_all_to_all'))


Formulation: sparse_topology
Target compression: 0.4
Beta used: 0.779
Beta auto calibrated: True
Interaction method: same_stage
Sparse ZZ terms: 12
Legacy all-to-all ZZ terms for same n: 45
ZZ reduction vs legacy all-to-all: 0.7333333333333334


## Inspect best classical solution

This is only a brute-force sanity check for the generated QUBO. The QAOA notebook should be rerun separately after this file is generated.


In [4]:
import csv

if not energy_check_path.exists():
    raise FileNotFoundError('Missing qubo_energy_check.csv. The generator may have failed before creating the energy check.')

with energy_check_path.open('r', encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f))

best = rows[0]

print('Best bitstring:', best.get('bitstring'))
print('Energy:', best.get('energy'))
print('Compression:', best.get('compression'))
print('Loss penalty:', best.get('loss_penalty'))
print('Pruned blocks:', best.get('pruned_blocks'))


Best bitstring: 1100000100
Energy: -0.26490372822367836
Compression: 0.3919314581933378
Loss penalty: 0.04041087770893179
Pruned blocks: stages.2.blocks.2; stages.3.blocks.2; stages.1.blocks.1


## Generated files

These files are generated automatically and normally should not be manually edited:

- `qubo_outputs/selected_candidates.csv`
- `qubo_outputs/interaction_edges.csv`
- `qubo_outputs/qubo_matrix.csv`
- `qubo_outputs/qubo_terms.json`
- `qubo_outputs/hamiltonian_terms.json`
- `qubo_outputs/qubo_metadata.json`
- `qubo_outputs/circuit_complexity_report.json`
- `qubo_outputs/qubo_energy_check.csv`

After this notebook runs, rerun `qaoa_experiment.ipynb` using the newly generated `qubo_outputs/hamiltonian_terms.json`.
